# Routing clinical notes by specialty with Claude

Classifies 50 medical transcriptions into one of six specialties and extracts the chief complaint, using one prompt with a system prompt, few-shot examples, XML-tagged input and output, and an optional reasoning step.

**Data:** MTSamples (mtsamples.com), via the Kaggle "Medical Transcriptions" dataset by Tara Boyle, CC0. Sample, de-identified transcriptions.
**Model:** claude-haiku-5-5

In [1]:
import re
import anthropic
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)
client = anthropic.Anthropic()  # retries rate-limit and server errors twice automatically
MODEL_NAME = "claude-haiku-5-5"
pd.set_option("display.max_colwidth", 120)

def get_completion(prompt, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        system=system_prompt,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in message.content if b.type == "text")

SPECIALTIES = ["Cardiovascular / Pulmonary", "Orthopedic", "Gastroenterology",
               "Neurology", "Urology", "Obstetrics / Gynecology"]

In [2]:
SYSTEM_PROMPT = f"""You are a clinical documentation specialist who routes medical transcriptions.

Your job: for one transcription, identify
1. specialty: exactly one of: {", ".join(SPECIALTIES)}
2. chief_complaint: the main reason for the encounter, in a short phrase taken from the note
3. reason: one sentence citing the specific details in the note that decided the specialty

Rules:
- Use only what the note says. Do not add diagnoses or findings that are not in the note.
- Copy the specialty name exactly as written in the list above.
- If a note touches several areas, choose the specialty of the main procedure or consult performed.

Examples of the exact output format:
<examples>
<example>
<record>PROCEDURE: , Gastroscopy.,PREOPERATIVE DIAGNOSES: , Dysphagia, possible stricture.</record>
<answer><specialty>Gastroenterology</specialty><chief_complaint>dysphagia, possible stricture</chief_complaint><reason>The main procedure is a gastroscopy to evaluate difficulty swallowing and a possible stricture.</reason></answer>
</example>
<example>
<record>CT ANGIOGRAPHY CHEST WITH CONTRAST,REASON FOR EXAM:  ,Shortness of breath for two weeks and a history of pneumonia.  The patient also has a history of left lobectomy.</record>
<answer><specialty>Cardiovascular / Pulmonary</specialty><chief_complaint>shortness of breath for two weeks</chief_complaint><reason>A chest CT angiogram for shortness of breath in a patient with prior pneumonia and lobectomy is a cardiopulmonary workup.</reason></answer>
</example>
</examples>"""

In [3]:
def build_prompt(note_text, think=False):
    if think:
        instruction = ("Analyze the medical transcription in the <record> tags. First, reason step by "
                       "step inside <thinking> tags: note the body system involved, the main procedure "
                       "or consult, and the reason for the encounter. Then give your final answer in an "
                       "<answer> block with <specialty>, <chief_complaint>, and <reason>, exactly as in "
                       "the examples.")
    else:
        instruction = ("Analyze the medical transcription in the <record> tags. Respond with only an "
                       "<answer> block with <specialty>, <chief_complaint>, and <reason>, exactly as in "
                       "the examples.")
    return f"<record>\n{note_text}\n</record>\n\n{instruction}"

def extract(tag, text):
    match = re.search(f"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return match.group(1).strip() if match else None

In [4]:
data = pd.read_csv("data/mtsamples_subset.csv")

# Same numbers as prompt_practice, so the same rows get excluded
practice = data.sample(8, random_state=1)
example_pool = data.drop(practice.index).sample(6, random_state=2)
remaining = data.drop(practice.index).drop(example_pool.index)

df = remaining.sample(50, random_state=3).reset_index(drop=True)
print(df.shape)
print(df["medical_specialty"].value_counts())

(50, 2)
medical_specialty
Neurology                     10
Cardiovascular / Pulmonary     9
Gastroenterology               8
Urology                        8
Orthopedic                     8
Obstetrics / Gynecology        7
Name: count, dtype: int64


In [5]:
USE_THINKING = False  # test record before committing to full build

for _, row in df.head(3).iterrows():
    out = get_completion(build_prompt(row["transcription"], think=USE_THINKING), SYSTEM_PROMPT)
    print(row["medical_specialty"], "→", extract("specialty", out))

Gastroenterology → Gastroenterology
Cardiovascular / Pulmonary → Cardiovascular / Pulmonary
Cardiovascular / Pulmonary → Cardiovascular / Pulmonary


In [6]:
raw_outputs = []
for n, (_, row) in enumerate(df.iterrows(), start=1):
    try:
        out = get_completion(build_prompt(row["transcription"], think=USE_THINKING), SYSTEM_PROMPT)
    except Exception as e:
        out = f"ERROR: {e}"   # one failed row doesn't stop the run
    raw_outputs.append(out)
    print(f"\r{n}/{len(df)} done", end="")

df["raw_output"] = raw_outputs

50/50 done

In [7]:
for tag in ["specialty", "chief_complaint", "reason"]:
    df[f"pred_{tag}"] = df["raw_output"].apply(lambda text: extract(tag, text))

df["correct"] = df["pred_specialty"] == df["medical_specialty"]
df[["medical_specialty", "pred_specialty", "pred_chief_complaint", "correct"]]

,medical_specialty,pred_specialty,pred_chief_complaint,correct
0,Gastroenterology,Gastroenterology,rectal bleeding,True
1,Cardiovascular / Pulmonary,Cardiovascular / Pulmonary,acute respiratory failure with complete left lung atelectasis after multiple trauma/motor vehicle accident,True
2,Cardiovascular / Pulmonary,Cardiovascular / Pulmonary,empyema thoracis requiring intravenous antibiotics,True
3,Urology,Urology,elective male sterilization via bilateral vasectomy,True
4,Obstetrics / Gynecology,Obstetrics / Gynecology,"advanced maternal age and hypertension, pregnancy surveillance at 36 weeks 4 days",True
5,Neurology,Neurology,"recurrent early-morning myoclonic jerking in juvenile myoclonic epilepsy, with a recent generalized tonic-clonic sei...",True
6,Urology,Urology,elective neonatal circumcision,True
7,Orthopedic,Orthopedic,right shoulder pain,True
8,Cardiovascular / Pulmonary,Cardiovascular / Pulmonary,"left lower lobe lung mass with sarcomatoid carcinoma, status post left lower lobe resection",True
9,Cardiovascular / Pulmonary,Cardiovascular / Pulmonary,"unstable angina, atherosclerotic coronary artery disease",True


In [8]:
print("Accuracy:        ", df["correct"].mean())
print("Format failures: ", df["pred_specialty"].isna().sum())
print("API errors:      ", df["raw_output"].str.startswith("ERROR").sum())

pd.crosstab(df["medical_specialty"], df["pred_specialty"])

Accuracy:         0.94
Format failures:  0
API errors:       0


pred_specialty,Cardiovascular / Pulmonary,Gastroenterology,Neurology,Obstetrics / Gynecology,Orthopedic,Unable to determine,Urology
medical_specialty,,,,,,,
Cardiovascular / Pulmonary,9,0,0,0,0,0,0
Gastroenterology,0,8,0,0,0,0,0
Neurology,1,0,9,0,0,0,0
Obstetrics / Gynecology,0,0,0,7,0,0,0
Orthopedic,0,0,1,0,7,0,0
Urology,0,0,0,0,0,1,7


In [9]:
misses = df[~df["correct"]]
for i, row in misses.iterrows():
    print(f"===== Row {i} =====")
    print("Dataset label:", row["medical_specialty"])
    print("Claude said:  ", row["pred_specialty"])
    print("Reason:       ", row["pred_reason"])
    print("Note start:   ", row["transcription"][:800])
    print()

===== Row 13 =====
Dataset label: Urology
Claude said:   Unable to determine
Reason:        The record contains only the heading "CHIEF COMPLAINT:" with no content, so no specialty, complaint, or supporting details can be identified from the note.
Note start:    CHIEF COMPLAINT:,

===== Row 39 =====
Dataset label: Orthopedic
Claude said:   Neurology
Reason:        The encounter centers on neurologic decline in a patient with a prior right frontal glioblastoma multiforme, evidenced by proximal lower extremity weakness (4+/5), gait difficulty, abnormal reflex and motor exam findings, an MRI L-spine for suspected leptomeningeal "drop" metastasis, and CSF analysis.
Note start:    CC:, Progressive lower extremity weakness.,HX: ,This 52y/o RHF had a h/o right frontal glioblastoma multiforme (GBM) diagnosed by brain biopsy/partial resection, on 1/15/1991. She had been healthy until 1/6/91, when she experienced a generalized tonic-clonic type seizure during the night. She subsequently underwen

In [10]:
df.to_csv("data/mtsamples_50_routed.csv", index=False)
print("Saved", len(df), "rows")

Saved 50 rows


## Results
- Accuracy: 47 of 50 (94%)
- Mismatches:
  - Row 13: labeled Urology, Claude said unable to determine — Claude likely correct as the record contains only a heading.
  - Row 39: labeled Orthopedic, Claude said Neurology - Claude likely correct b/c MRI brain scan noted a right frontal lesion that was removed.
  - Row 44: labeled Neurology, Claude said Cardiovascular/Pulmonary - Claude likely correct b/c no speciality listed for the main request and Cardiovascular/Pulmonary is the closest speciality from the list.
- What I'd try next: Flag uncertain notes for human review. Add a <confidence> tag (high/low) and route low-confidence notes to a person instead of auto-assigning them.